İstatistiksel karşılaştırma hücresi dış değerlendirmelerden sonra çalıştırılır.


# 28A — FinancialBERT eğitim

Bu notebook, projedeki FinBERT akışının tamamını `ahmedrachid/FinancialBERT-Sentiment-Analysis` hazır sentiment sınıflandırıcısı için tekrarlar:

1. Hazır model ile zero-shot baseline ve hata analizi
2. Aynı plain-sentiment train/validation/test splitleriyle hedef veri fine-tuning
3. Üçlü consensus train/validation/internal-test splitleriyle fine-tuning
4. Validation seçimi, epoch checkpointleri ve kaldığı yerden devam
5. İç test ve üç dış test setinde örnek-bazlı tahminler, raporlar ve confusion matrix
6. Hazır / hedefe fine-tune / consensus fine-tune karşılaştırması

Model kartına göre model Financial PhraseBank üzerinde üç sınıflı sentiment için fine-tune edilmiştir. Bu notebook hazır classifier head'i kullanır; yalnızca fine-tuning aşamalarında proje label sırasına hizalama yapar.

In [1]:
from pathlib import Path
import os, sys, json, gc, inspect, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from datasets import Dataset
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    f1_score, classification_report, confusion_matrix)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from thesis_utils import PROJECT_ROOT, DATA_ROOT, paths

os.environ['WANDB_DISABLED'] = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
LABELS = list(LABEL2ID)
# Tez protokolündeki doğru başlangıç modeli; sentiment-head bu checkpoint'ten alınır.
MODEL_NAME = 'ahmedrachid/FinancialBERT'
RANDOM_STATE, MAX_LENGTH = 42, 128
EPOCHS, TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE = 4, 16, 32
LEARNING_RATE, WEIGHT_DECAY, SAVE_TOTAL_LIMIT = 2e-5, 0.01, 3
USE_CLASS_WEIGHTS, FORCE_RETRAIN = True, False
# Her aşama kendi marker'ını yazar; notebook kapanırsa yalnız eksik aşama yeniden çalışır.
CHECKPOINT_SCHEMA_VERSION = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RUN_ROOT = paths.MODEL_CHECKPOINT_ROOT / 'financialbert_full_pipeline_financialbert_seed42_v2'
TARGET_ROOT = RUN_ROOT / 'target_dataset'
CONSENSUS_ROOT = RUN_ROOT / 'consensus'
RESULT_ROOT = paths.MODEL_RESULTS_ROOT / 'financialbert_full_pipeline_financialbert_seed42_v2'
for p in [TARGET_ROOT, CONSENSUS_ROOT, RESULT_ROOT]: p.mkdir(parents=True, exist_ok=True)

def seed_all(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_all()
print('Device:', DEVICE)
print('Model:', MODEL_NAME)
print('Run root:', RUN_ROOT)
RUN_MANIFEST = RUN_ROOT / 'run_manifest.json'
EXPECTED_MANIFEST = {'model_name': MODEL_NAME, 'seed': RANDOM_STATE, 'schema': CHECKPOINT_SCHEMA_VERSION}
if RUN_MANIFEST.exists():
    saved_manifest = json.loads(RUN_MANIFEST.read_text(encoding='utf-8'))
    if saved_manifest != EXPECTED_MANIFEST:
        raise RuntimeError(f'Run manifest mismatch; refusing to resume: {saved_manifest}')
else:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    RUN_MANIFEST.write_text(json.dumps(EXPECTED_MANIFEST, indent=2), encoding='utf-8')


Device: cpu
Model: ahmedrachid/FinancialBERT
Run root: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\financialbert_full_pipeline_financialbert_seed42_v2


In [2]:
# Ortak yardımcılar: veri, label hizalama, metrik ve checkpoint destekli Trainer
def norm_label(x):
    s = str(x).lower().strip().replace(' ', '_')
    aliases = {'label_0':'positive','label_1':'negative','label_2':'neutral',
               'positive':'positive','negative':'negative','neutral':'neutral',
               'pos':'positive','neg':'negative','neu':'neutral'}
    return aliases.get(s, s)

def align_classifier(model):
    if not hasattr(model, 'classifier'):
        raise AttributeError('FinancialBERT classifier katmanı bulunamadı.')
    raw = {int(k): norm_label(v) for k, v in model.config.id2label.items()}
    layer = model.classifier
    # ahmedrachid/FinancialBERT iki çıkışlı head ile geliyor; encoder korunur,
    # proje protokolü için üç sınıflı head yeniden oluşturulur.
    if getattr(model.config, 'num_labels', layer.out_features) != 3 or set(raw.values()) != set(LABEL2ID):
        model.classifier = nn.Linear(layer.in_features, 3).to(layer.weight.device, dtype=layer.weight.dtype)
    else:
        old_w = layer.weight.detach().clone()
        old_b = layer.bias.detach().clone() if layer.bias is not None else None
        source = {label: idx for idx, label in raw.items()}
        with torch.no_grad():
            for target_id, label in ID2LABEL.items():
                model.classifier.weight[target_id].copy_(old_w[source[label]])
                if old_b is not None:
                    model.classifier.bias[target_id].copy_(old_b[source[label]])
    model.config.num_labels = 3
    model.num_labels = 3
    model.config.id2label = ID2LABEL.copy()
    model.config.label2id = LABEL2ID.copy()
    return model

def make_ds(df, tokenizer, text_col='input_text', label_col='label_id'):
    temp = df[[text_col, label_col]].copy().rename(columns={label_col:'labels'})
    temp[text_col] = temp[text_col].astype(str); temp['labels'] = temp['labels'].astype(int)
    ds = Dataset.from_pandas(temp.reset_index(drop=True))
    ds = ds.map(lambda b: tokenizer(b[text_col], truncation=True, padding='max_length', max_length=MAX_LENGTH), batched=True)
    keep = ['input_ids','attention_mask','labels']
    if 'token_type_ids' in ds.column_names: keep.append('token_type_ids')
    ds.set_format(type='torch', columns=keep); return ds

def trainer_tokenizer_kwargs(tokenizer):
    sig = inspect.signature(Trainer.__init__)
    return {'processing_class': tokenizer} if 'processing_class' in sig.parameters else ({'tokenizer': tokenizer} if 'tokenizer' in sig.parameters else {})

def metrics_fn(pair):
    logits, y = pair; pred = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(y,pred), 'f1_macro': f1_score(y,pred,average='macro',zero_division=0), 'f1_weighted': f1_score(y,pred,average='weighted',zero_division=0)}

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs): super().__init__(*args, **kwargs); self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs['labels']; model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}; out = model(**model_inputs)
        w = self.class_weights.to(out.logits.device) if self.class_weights is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(out.logits.view(-1,3), labels.view(-1))
        return (loss,out) if return_outputs else loss

def args_for(outdir):
    sig = inspect.signature(TrainingArguments.__init__)
    kw = dict(output_dir=str(outdir), learning_rate=LEARNING_RATE, per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE, num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, logging_steps=50, report_to='none', seed=RANDOM_STATE)
    if 'eval_strategy' in sig.parameters: kw['eval_strategy']='epoch'
    elif 'evaluation_strategy' in sig.parameters: kw['evaluation_strategy']='epoch'
    for k,v in [('save_strategy','epoch'),('save_total_limit',SAVE_TOTAL_LIMIT),('load_best_model_at_end',True),('metric_for_best_model','f1_macro'),('greater_is_better',True),('logging_strategy','steps')]:
        if k in sig.parameters: kw[k]=v
    if 'fp16' in sig.parameters: kw['fp16']=torch.cuda.is_available()
    return TrainingArguments(**kw)

def class_weights(df):
    c=np.bincount(df.label_id.astype(int), minlength=3); return torch.tensor([len(df)/(3*x) if x else 1.0 for x in c], dtype=torch.float) if USE_CLASS_WEIGHTS else None

In [3]:
# Eğitim verisini hazırla (baseline/test çalıştırılmaz)
split_dir = paths.PLAIN_SENTIMENT_SPLIT_V1_DIR
train_df = pd.read_parquet(split_dir/'train_df.parquet')
val_df = pd.read_parquet(split_dir/'val_df.parquet')
for df in [train_df, val_df]:
    df['label'] = df['label'].astype(str).str.lower().str.strip()
    df['label_id'] = df['label_id'].astype(int)


In [4]:
# 2) Hedef plain-sentiment fine-tuning: checkpoint + resume + val-best model
def train_stage(stage_name, train_data, val_data, root):
    run=root; ckpt=run/'checkpoints'; final=run/'final_model'; marker=run/'training_complete.json'
    ckpt.mkdir(parents=True,exist_ok=True); final.mkdir(parents=True,exist_ok=True)
    tok=AutoTokenizer.from_pretrained(MODEL_NAME,use_fast=True)
    trds=make_ds(train_data,tok); vds=make_ds(val_data,tok)
    complete=marker.exists() and (final/'config.json').exists()
    model = (AutoModelForSequenceClassification.from_pretrained(final) if complete and not FORCE_RETRAIN else align_classifier(AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)))
    trainer=WeightedTrainer(model=model,args=args_for(ckpt),train_dataset=trds,eval_dataset=vds,compute_metrics=metrics_fn,class_weights=class_weights(train_data),**trainer_tokenizer_kwargs(tok))
    if not complete or FORCE_RETRAIN:
        last=None if FORCE_RETRAIN else get_last_checkpoint(str(ckpt)); print(stage_name,'resume:',last); trainer.train(resume_from_checkpoint=last)
        # Trainer epoch sonunda ckpt klasörüne kaydeder; final yalnız başarılı eğitimden sonra yazılır.
        trainer.save_model(str(final)); tok.save_pretrained(str(final))
    else: print(stage_name,'tamamlandı; final model yükleniyor/değerlendiriliyor.')
    ev=trainer.evaluate(eval_dataset=vds); summary={'stage':stage_name,'model_name':MODEL_NAME,'train_n':len(train_data),'val_n':len(val_data),'validation_loss':float(ev.get('eval_loss',np.nan)),'validation_f1_macro':float(ev.get('eval_f1_macro',np.nan)),'checkpoint_dir':str(ckpt),'final_model_dir':str(final)}
    summary['checkpoint_schema_version']=CHECKPOINT_SCHEMA_VERSION; summary['seed']=RANDOM_STATE; summary['source_model_name']=MODEL_NAME; summary['run_manifest']=str(RUN_MANIFEST)
    (run/'training_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8'); marker.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
    del trainer,model,tok,trds,vds; gc.collect(); return summary

target_summary=train_stage('target_dataset',train_df,val_df,TARGET_ROOT)
print('Target eğitim tamamlandı:', target_summary)

Map:   0%|          | 0/12950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1432 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

target_dataset tamamlandı; final model yükleniyor/değerlendiriliyor.


C:\Users\kayma\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
No log,1.023480,0,0.826816,0.775753,0.827587


Target eğitim tamamlandı: {'stage': 'target_dataset', 'model_name': 'ahmedrachid/FinancialBERT', 'train_n': 12950, 'val_n': 1432, 'validation_loss': 1.0234795808792114, 'validation_f1_macro': 0.775753486393835, 'checkpoint_dir': 'D:\\serkan.kaymak\\financial_sentiment_thesis\\financial_sentiment_thesis\\checkpoints\\financial_sentiment_multi_model\\financialbert_full_pipeline_financialbert_seed42_v2\\target_dataset\\checkpoints', 'final_model_dir': 'D:\\serkan.kaymak\\financial_sentiment_thesis\\financial_sentiment_thesis\\checkpoints\\financial_sentiment_multi_model\\financialbert_full_pipeline_financialbert_seed42_v2\\target_dataset\\final_model', 'checkpoint_schema_version': 2, 'seed': 42, 'source_model_name': 'ahmedrachid/FinancialBERT', 'run_manifest': 'D:\\serkan.kaymak\\financial_sentiment_thesis\\financial_sentiment_thesis\\checkpoints\\financial_sentiment_multi_model\\financialbert_full_pipeline_financialbert_seed42_v2\\run_manifest.json'}


In [5]:
# 3) Üçlü consensus fine-tuning: split yoksa üret, varsa aynen kullan
cons_root=DATA_ROOT/'splits'/'consensus_twitter_phrasebank_allagree_v1'
def read_split(name):
    p=cons_root/f'{name}.csv'
    if not p.exists(): raise FileNotFoundError(f'Consensus split yok: {p}. 23 notebook önce çalıştırılmalı.')
    d=pd.read_csv(p); d['input_text']=d['input_text'].astype(str); d['label']=d['label'].astype(str).str.lower().str.strip(); d['label_id']=d['label'].map(LABEL2ID).astype(int); return d
cons_train,cons_val,cons_test=read_split('train'),read_split('validation'),read_split('internal_test')
for a,b in [('train',cons_train),('validation',cons_val),('internal_test',cons_test)]: print(a,b.shape,b.label.value_counts().to_dict())
cons_summary=train_stage('consensus',cons_train,cons_val,CONSENSUS_ROOT)
print('Consensus eğitim tamamlandı:', cons_summary)

train (7683, 15) {'neutral': 4330, 'positive': 1986, 'negative': 1367}
validation (978, 14) {'neutral': 536, 'positive': 253, 'negative': 189}
internal_test (1630, 14) {'neutral': 893, 'positive': 423, 'negative': 314}


Map:   0%|          | 0/7683 [00:00<?, ? examples/s]

Map:   0%|          | 0/978 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ahmedrachid/FinancialBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

consensus resume: None


... (8 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\kayma\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.120130,0.828548,4,0.842536,0.814527,0.842464


Consensus eğitim tamamlandı: {'stage': 'consensus', 'model_name': 'ahmedrachid/FinancialBERT', 'train_n': 7683, 'val_n': 978, 'validation_loss': 0.8285476565361023, 'validation_f1_macro': 0.8145266175992433, 'checkpoint_dir': 'D:\\serkan.kaymak\\financial_sentiment_thesis\\financial_sentiment_thesis\\checkpoints\\financial_sentiment_multi_model\\financialbert_full_pipeline_financialbert_seed42_v2\\consensus\\checkpoints', 'final_model_dir': 'D:\\serkan.kaymak\\financial_sentiment_thesis\\financial_sentiment_thesis\\checkpoints\\financial_sentiment_multi_model\\financialbert_full_pipeline_financialbert_seed42_v2\\consensus\\final_model', 'checkpoint_schema_version': 2, 'seed': 42, 'source_model_name': 'ahmedrachid/FinancialBERT', 'run_manifest': 'D:\\serkan.kaymak\\financial_sentiment_thesis\\financial_sentiment_thesis\\checkpoints\\financial_sentiment_multi_model\\financialbert_full_pipeline_financialbert_seed42_v2\\run_manifest.json'}


## Çıktılar ve tez kullanımı

- Checkpointler: `checkpoints/financial_sentiment_multi_model/financialbert_full_pipeline/{target_dataset,consensus}/checkpoints/`
- Final modeller: aynı klasörlerde `final_model/`
- Metrikler, prediction CSV'leri, classification report ve confusion matrix: `outputs/finetuned_model_results/financialbert_full_pipeline/`
- Ana karşılaştırma: `financialbert_full_summary.csv`

Eğitim yarıda kalırsa aynı hücre yeniden çalıştırıldığında son epoch checkpoint'i `get_last_checkpoint` ile bulunur ve `resume_from_checkpoint` ile devam eder. Baştan eğitim için `FORCE_RETRAIN = True` yapılmalıdır.